# Task 6: JSONPlaceholder Posts에서 짝수 인덱스 데이터를 엑셀로 저장하기

## 과제 목표
1. `https://jsonplaceholder.typicode.com/posts`에서 데이터를 불러온다.
2. **index가 짝수인 내용만** 필터링한다.
3. 필터링된 데이터를 **엑셀(.xlsx) 파일로 저장**하여 다운로드 받는다.

## 학습 목표
1. REST API에서 대량의 JSON 데이터를 수집하는 방법을 복습한다.
2. pandas의 **인덱싱(indexing)** 방법 3가지를 비교·이해한다.
3. DataFrame을 **엑셀 파일**로 저장하는 방법과 `openpyxl` 엔진의 역할을 배운다.

---
## 1단계: 환경 설정

### 왜 `openpyxl`이 필요한가?

pandas의 `to_excel()` 메서드는 **자체적으로 엑셀 파일을 생성하지 못합니다.**
엑셀 파일(.xlsx)의 복잡한 바이너리 구조를 처리하기 위해 **외부 엔진**이 필요합니다.

| 엔진 | 지원 형식 | 특징 |
|------|----------|------|
| `openpyxl` | `.xlsx` (Excel 2010+) | 가장 널리 사용, pandas 기본 엔진 |
| `xlsxwriter` | `.xlsx` | 차트/서식 지원이 뛰어남 |
| `xlwt` | `.xls` (구형) | 레거시 Excel 97-2003 형식 |

pandas는 `to_excel()` 호출 시 자동으로 `openpyxl`을 찾아 사용합니다.
만약 설치되어 있지 않으면 `ModuleNotFoundError`가 발생합니다.

In [ ]:
import requests
import pandas as pd
import os

# openpyxl 설치 확인
try:
    import openpyxl
    print(f"openpyxl 버전: {openpyxl.__version__} ✅")
except ImportError:
    print("openpyxl이 설치되어 있지 않습니다. 아래 명령으로 설치하세요:")
    print("  pip install openpyxl")

print("환경 설정 완료!")

---
## 2단계: API에서 Posts 데이터 수집

### `/posts` 엔드포인트의 데이터 구조

이 API는 100개의 가짜 게시글 데이터를 반환합니다.
각 게시글의 구조:

```json
{
  "userId": 1,      ← 작성자 ID (1~10)
  "id": 1,           ← 게시글 고유 ID (1~100)
  "title": "...",    ← 제목
  "body": "..."      ← 본문
}
```

Task 5의 `/users`와 달리 **중첩 구조가 없는 단순(flat) JSON**입니다.
따라서 `pd.DataFrame()`만으로 바로 변환이 가능합니다.

### 상태 코드 검증의 중요성

실무에서는 API 호출 후 반드시 **상태 코드를 확인**해야 합니다:
- `200`: 성공
- `404`: 리소스를 찾을 수 없음 (URL 오류)
- `500`: 서버 내부 오류

상태 코드를 무시하면 빈 데이터나 에러 메시지를 정상 데이터로 처리하는 **사일런트 에러(silent error)**가 발생할 수 있습니다.

In [ ]:
# API 호출
url = "https://jsonplaceholder.typicode.com/posts"
response = requests.get(url)

# 상태 코드 확인
print(f"상태 코드: {response.status_code}")

# JSON → DataFrame 변환
posts_data = response.json()
df_posts = pd.DataFrame(posts_data)

print(f"DataFrame 크기: {df_posts.shape}  ({df_posts.shape[0]}개 게시글, {df_posts.shape[1]}개 컬럼)")
print(f"컬럼: {list(df_posts.columns)}")
print(f"인덱스 범위: {df_posts.index[0]} ~ {df_posts.index[-1]}")
print()
df_posts.head()

---
## 3단계: 짝수 인덱스 데이터 필터링

### "인덱스가 짝수"란 무엇인가?

DataFrame의 인덱스는 기본적으로 `0, 1, 2, 3, ...`의 정수입니다.
"짝수 인덱스"란 `0, 2, 4, 6, ...`에 해당하는 행을 의미합니다.

### 3가지 필터링 방법 비교

| 방법 | 코드 | 원리 | 적합한 상황 |
|------|------|------|------------|
| **불리언 마스킹** | `df[df.index % 2 == 0]` | 조건식으로 True/False 배열 생성 후 필터 | 조건이 명확할 때 (가독성 최고) |
| **iloc 슬라이싱** | `df.iloc[::2]` | 위치 기반으로 2칸씩 건너뛰기 | 위치 기반 패턴이 규칙적일 때 |
| **loc 슬라이싱** | `df.loc[::2]` | 라벨(인덱스 값) 기반 슬라이싱 | 인덱스가 정수이고 정렬되어 있을 때 |

### `%` (모듈로) 연산자란?

`%`는 **나머지 연산자**입니다. `n % 2`는:
- `n`이 짝수면 → `0` (2로 나누어 떨어짐)
- `n`이 홀수면 → `1` (나머지가 1)

따라서 `df.index % 2 == 0`은 "인덱스를 2로 나눈 나머지가 0인가?" → "짝수인가?"와 같습니다.

### 왜 불리언 마스킹을 선택하는가?

세 방법 모두 같은 결과를 주지만, `df[df.index % 2 == 0]`은:
1. **의도가 코드에 명확히 드러남**: "인덱스가 짝수인 행을 선택"
2. **인덱스가 재정렬되어도 안전**: `iloc[::2]`는 위치만 보지만, `index % 2 == 0`은 실제 인덱스 값을 검사

In [ ]:
# 방법 1: 불리언 마스킹 (권장)
df_even = df_posts[df_posts.index % 2 == 0]

print(f"원본 DataFrame: {len(df_posts)}행")
print(f"짝수 인덱스 필터링 후: {len(df_even)}행")
print(f"인덱스 목록 (앞 10개): {list(df_even.index[:10])}")
print()

# 결과 확인
df_even.head(10)

In [ ]:
# (참고) 세 가지 방법의 결과가 동일한지 검증
method1 = df_posts[df_posts.index % 2 == 0]     # 불리언 마스킹
method2 = df_posts.iloc[::2]                      # iloc 슬라이싱
method3 = df_posts.loc[::2]                       # loc 슬라이싱

print(f"방법1 == 방법2: {method1.equals(method2)}")
print(f"방법1 == 방법3: {method1.equals(method3)}")
print("→ 세 방법 모두 동일한 결과!")

---
## 4단계: 엑셀 파일로 저장

### `to_excel()` 함수의 주요 파라미터

```python
df.to_excel(filepath, sheet_name='Sheet1', index=True, engine='openpyxl')
```

| 파라미터 | 기본값 | 설명 |
|----------|--------|------|
| `filepath` | (필수) | 저장할 파일 경로. `.xlsx` 확장자 사용 |
| `sheet_name` | `'Sheet1'` | 시트 이름 |
| `index` | `True` | DataFrame의 인덱스를 파일에 포함할지 여부 |
| `engine` | `'openpyxl'` | 엑셀 파일 생성에 사용할 엔진 |

### `index=False`로 설정하는 이유

기본값(`index=True`)으로 저장하면 엑셀 파일에 **인덱스가 별도 열로 추가**됩니다:

| (인덱스) | userId | id | title | body |
|:--------:|--------|----|----|------|
| 0 | 1 | 1 | ... | ... |

`index=False`로 설정하면 불필요한 인덱스 열이 제거되어 **깔끔한 데이터**만 남습니다.

### 파일 경로 설정

현재 노트북이 실행되는 디렉토리에 저장합니다.
`os.path.abspath()`로 절대 경로를 확인하면 파일을 찾기 쉽습니다.

In [ ]:
# 엑셀 파일로 저장
output_filename = "posts_even_index.xlsx"

df_even.to_excel(
    output_filename,
    sheet_name="짝수인덱스",    # 시트 이름을 한국어로 지정
    index=False,               # 인덱스 열 제외
    engine="openpyxl"          # 명시적으로 엔진 지정
)

# 저장 확인
file_path = os.path.abspath(output_filename)
file_size = os.path.getsize(output_filename)

print(f"엑셀 파일 저장 완료! ✅")
print(f"파일명: {output_filename}")
print(f"절대 경로: {file_path}")
print(f"파일 크기: {file_size:,} bytes ({file_size/1024:.1f} KB)")
print(f"저장된 행 수: {len(df_even)}행")

---
## 5단계: 저장된 엑셀 파일 검증

### 왜 검증이 필요한가?

파일이 생성되었다고 해서 데이터가 올바르게 저장된 것은 아닙니다.
실무에서는 **"쓰기 → 읽기 → 비교"** 패턴으로 데이터 무결성을 검증합니다.

이를 **라운드트립 검증(round-trip verification)**이라고 합니다.

### `pd.read_excel()`로 재검증

저장한 엑셀 파일을 다시 읽어 원본 DataFrame과 비교합니다.

In [ ]:
# 저장된 엑셀 파일을 다시 읽어서 검증
df_verify = pd.read_excel(output_filename, sheet_name="짝수인덱스", engine="openpyxl")

print(f"읽어온 DataFrame 크기: {df_verify.shape}")
print(f"원본과 행 수 일치: {len(df_verify) == len(df_even)} ({'✅' if len(df_verify) == len(df_even) else '❌'})")
print(f"원본과 컬럼 일치: {list(df_verify.columns) == list(df_even.columns)} ({'✅' if list(df_verify.columns) == list(df_even.columns) else '❌'})")
print()
df_verify.head()

---
## 핵심 정리

### 이번 과제에서 배운 것

| 개념 | 핵심 내용 |
|------|----------|
| **짝수 인덱스 필터링** | `df[df.index % 2 == 0]` — 불리언 마스킹이 가장 명확 |
| **`to_excel()`** | `openpyxl` 엔진을 사용하여 `.xlsx` 파일 생성 |
| **`index=False`** | 불필요한 인덱스 열을 엑셀에 포함하지 않음 |
| **라운드트립 검증** | 저장 후 `read_excel()`로 다시 읽어 데이터 무결성 확인 |

### 핵심 코드 요약

```python
# 1. API → DataFrame
df = pd.DataFrame(requests.get(url).json())

# 2. 짝수 인덱스 필터
df_even = df[df.index % 2 == 0]

# 3. 엑셀 저장
df_even.to_excel('output.xlsx', index=False)
```